# Setup

## ETL

For ETL, we directly call the appropriate `dlctl` commands for:

1. Ingesting the dataset
2. Transforming using SQL on top of DuckLake
3. Exporting from the data lakehouse into Parquet
4. Loading the graph into Kuzu
5. Computing general analytics scores

Be sure to uncomment the cell below and run it once.

In [ ]:
# # Uncomment and run once
# !dlctl ingest dataset -t atlas "The Atlas of Economic Complexity"
# !dlctl transform -m +marts.graphs.econ_comp
# !dlctl export dataset graphs econ_comp
# !dlctl graph load econ_comp
# !dlctl graph compute con-score econ_comp Country CompetesWith

## Imports

In [ ]:
from pathlib import Path
from string import Template
from textwrap import dedent
from typing import Any, Literal, Optional

import kuzu
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy.special import expit

import graph.visualization as vis
from shared.settings import LOCAL_DIR, env

## Globals

We setup access to the appropriate Kuzu path, based on the shared `.env` configuration, ensuring the graph exists before running the notebook. Once setup, `conn` will be used to query the graph directly throughout this notebook.

In [ ]:
db_path = Path(LOCAL_DIR) / env.str("ECON_COMP_GRAPH_DB")
assert db_path.exists(), "You need to create the graph DB using dlctl first"

In [ ]:
db = kuzu.Database(db_path)
conn = kuzu.Connection(db)

## Constants

In order to ensure color consistency for our plots, we extract the color palette from `matplotlib` into `MPL_PALETTE`.

In [ ]:
MPL_PALETTE = plt.rcParams["axes.prop_cycle"].by_key()["color"]

We also map a display attribute for each of our note labels, `Country` and `Product`. We'll use the short names for both when plotting graph visualizations or related charts.

In [ ]:
LABEL_PROPS = {
    "Country": "country_name_short",
    "Product": "product_name_short",
}

## Functions

We create a few reusable functions, where we run Kuzu queries. In a few cases, it was helpful to debug the query with parameters (e.g., using Kuzu Explorer), so we created a helper function for this (note that this doesn't support string parameters, as we didn't ned them).

In [ ]:
def print_query(query: str, params: dict[str, Any]):
    dbg_query = dedent(query).strip()
    dbg_query = Template(dbg_query)
    dbg_query = dbg_query.substitute(params)
    print(dbg_query)

We'll also cluster nodes using different strategies and compare groups, so we implement a basic Jaccard similarity function.

In [ ]:
def jaccard_sim(a: pd.Series, b: pd.Series) -> float:
    a = set(a)
    b = set(b)
    return len(a & b) / len(a | b)

We might want to look at the top x% of traded products, based ona USD. The following function will help filter this.

In [ ]:
def top_frac(df: pd.DataFrame, col: str, frac: float = 0.25):
    mask = (df[col] / df[col].sum()).cumsum() <= frac
    return df[mask]

# Analysis

 We focus on the `CompetesWith` projection, a relationship given by the [Export Similarity Index (ESI)](https://www.federalreserve.gov/econres/notes/feds-notes/the-sectoral-evolution-of-chinas-trade-20250228.html). Our graph analysis includes:

1. Dynamic competition analysis.
   1. Dominating and weaker economy identification, based on the [CON score](https://arxiv.org/pdf/1909.06810) for each country.
   2. Trade basket overlap analysis for top and bottom economies.
2. Competition network analysis.
   1. Community analysis, including community mapping, top traded product identification, and trade alignment study (self-sufficiency, external competitiveness).
   2. Weak component analysis, following a similar approach to the community analysis—weak components widen community reach.
   3. Community and weak component comparison.
   4. Economical pressure analysis.

## Dynamic Competition Analysis

### Top 10 Dominating Economies

These are highly spread economies, able to compete with several other countries, i.e., with a high number of common out-neighbors (CON).

In [ ]:
dom_econ_df = conn.execute(
    """
    MATCH (c:Country)
    RETURN c, c.node_id AS node_id, c.country_name_short AS country
    ORDER BY c.con_score DESC
    LIMIT 10
    """
).get_as_df()[["node_id", "country"]]

dom_econ_df.index = pd.RangeIndex(start=1, stop=len(dom_econ_df) + 1, name="rank")
dom_econ_df

#### Top 3 Exports

Looking at the top exports will help contextualize these economies. We only look at the top 3 products, to keep the visualization clean and readable.

In [ ]:
dom_econ_g = conn.execute(
    """
    MATCH (c:Country)
    WITH c
    ORDER BY c.con_score DESC
    LIMIT 10

    MATCH (c)-[e:Exports]->(p:Product)
    MATCH (c2:Country)-[:Exports]->(p)

    WITH c, e, p, count(DISTINCT c2) AS exporters
    WHERE exporters > 1
    WITH c, e, p
    ORDER BY c.node_id, e.amount_usd DESC
    SKIP 0

    WITH c, collect({p: p, e: e}) AS export_list

    UNWIND list_slice(export_list, 0, 3) AS r
    RETURN c, r.e, r.p
    ORDER BY c.node_id, r.p.node_id
    """
).get_as_networkx()

vis.set_labels(dom_econ_g, LABEL_PROPS)
vis.plot(dom_econ_g, scale=1.25, seed=3)

### Bottom 10 Weaker Economies

These are smaller or weaker economies, in the sense that they have a lower competition power. We also find the `Undeclared` special country node at rank 1, showing that only a small number of products are undeclared worldwide.

In [ ]:
weak_econ_df = conn.execute(
    """
    MATCH (c:Country)
    RETURN c, c.node_id AS node_id, c.country_name_short AS country
    ORDER BY c.con_score ASC
    LIMIT 10
    """
).get_as_df()[["node_id", "country"]]

weak_econ_df.index = pd.RangeIndex(start=1, stop=len(weak_econ_df) + 1, name="rank")
weak_econ_df

#### Top 3 Exports

If we look at the top 3 exports for each competing country in the bottom of the ranking according CON scores, as expected we find that these are more disconnected economies, mostly focusing on raw materials, or components and machinery.

In [ ]:
weak_econ_g = conn.execute(
    """
    MATCH (c:Country)
    WITH c
    ORDER BY c.con_score ASC
    LIMIT 10

    MATCH (c)-[e:Exports]->(p:Product)
    MATCH (c2:Country)-[:Exports]->(p)

    WITH c, e, p, count(DISTINCT c2) AS exporters
    WHERE exporters > 1
    WITH c, e, p
    ORDER BY c.node_id, e.amount_usd DESC
    SKIP 0

    WITH c, collect({p: p, e: e}) AS export_list

    UNWIND list_slice(export_list, 0, 3) AS r
    RETURN c, r.e, r.p
    ORDER BY c.node_id, r.p.node_id
    """
).get_as_networkx()

vis.set_labels(weak_econ_g, LABEL_PROPS)
vis.plot(weak_econ_g, scale=1.25, seed=3)

### Dominating vs Weaker Economies

- Do dominating economies compete in the same markets as weaker economies?
  - If so, maybe that's why those weaker economies are being pushed to the bottom. ☑️
  - If not, maybe the products exported by those weaker economies are not the most competitive.

Here, we find that, due to the small export diversity, weaker economies are being crushed by dominating economies. Their position of vulnerability comes mostly from geographical isolation and limited area, leading to a lower amount of competition opportunities, where any competitor becomes a risk to the economy.

Below, country node classes are visually translated to a colored node border and label text. We assign two classes, for the top and bottom 10 economies, with top economies in the center, and the products and bottom economies in the surrounding area. This forms a star layout, where each arm is a weaker economy or a small cluster of weaker economies.

We look at the top 3 most exported products in weaker economies, but relaxing the filter on number of exported products for the weaker economies and looking at more than 3 exported products will reproduce the displayed behavior, with dominating economies still competing for the same products. This doesn't necessarily mean that both dominating and weaker economies produce the same products, as some of them can simply be re-exported.

In [ ]:
dom_vs_weak_econ_g = conn.execute(
    """
    MATCH (wea)-[we:Exports]->(p:Product)
    MATCH (dom)-[de:Exports]->(p)
    WHERE dom.node_id IN $dominating_node_ids
        AND wea.node_id IN $weaker_node_ids

    WITH wea, we, p, count(DISTINCT dom) AS dom_competitors
    WHERE dom_competitors > 0

    WITH wea, we, p
    ORDER BY wea.node_id, we.amount_usd DESC
    SKIP 0

    WITH wea, collect({p: p, e: we}) AS export_list
    UNWIND list_slice(export_list, 0, 3) AS r

    WITH wea, r.p.node_id AS prod_node_id
    MATCH (wea)-[we:Exports]->(prod:Product { node_id: prod_node_id })
    MATCH (dom:Country)-[de:Exports]->(prod)
    WHERE dom.node_id IN $dominating_node_ids
    RETURN wea, we, prod, de, dom
    ORDER BY wea.node_id, prod.node_id, dom.node_id
    """,
    dict(
        dominating_node_ids=dom_econ_df.node_id.to_list(),
        weaker_node_ids=weak_econ_df.node_id.to_list(),
    ),
).get_as_networkx()

node_classes = dict(
    dominating=dom_econ_df.node_id.to_list(),
    weaker=weak_econ_df.node_id.to_list(),
)

# This adjusts the visualization edge weights to improve readability
for u, v, data in dom_vs_weak_econ_g.edges(data=True):
    if (
        dom_vs_weak_econ_g.nodes[u]["node_id"] in node_classes["dominating"]
        and dom_vs_weak_econ_g.nodes[v]["_label"] == "Product"
    ):
        data["vis_weight"] = 1e-5

    if (
        dom_vs_weak_econ_g.nodes[u]["node_id"] in node_classes["weaker"]
        and dom_vs_weak_econ_g.nodes[v]["_label"] == "Product"
    ):
        data["vis_weight"] = 1e-3

vis.set_labels(dom_vs_weak_econ_g, LABEL_PROPS)
vis.plot(dom_vs_weak_econ_g, node_classes=node_classes, scale=1.25, seed=5)

## Competition Network

Let's look at the competition network projection for `Country` nodes and `CompetesWith` edges. We first install the `algo` extension for Kuzu and create the `compnet` projection and NetworkX graph for it.

In [ ]:
try:
    conn.execute(
        """
        INSTALL algo;
        LOAD algo;
        """
    )
except Exception as e:
    print(e)

In [ ]:
try:
    conn.execute(
        """
        CALL drop_projected_graph("compnet")
        """
    )
except Exception as e:
    print(e)

conn.execute(
    """
    CALL project_graph(
        "compnet",
        {"Country": "n.country_name_short <> 'Undeclared'"},
        {"CompetesWith": "true"}
    )
    """
)

In [ ]:
compnet_g = conn.execute(
    """
    MATCH (a:Country)-[cw:CompetesWith]->(b:Country)
    WHERE a.country_name_short <> "Undeclared"
        AND b.country_name_short <> "Undeclared"
    RETURN a, cw, b
    """,
).get_as_networkx()

### Inspection Functions

The following functions will be useful to plot the cluster and analyze the top exports for a specific cluster ID property:

In [ ]:
def plot_cluster(
    prop_name: str,
    prop_value: int,
    kind: Literal["graph", "map"] = "graph",
):
    match kind:
        case "graph":
            compnet_cluster_g = conn.execute(
                f"""
                MATCH (a:Country)-[cw:CompetesWith]->(b:Country)
                WHERE a.country_name_short <> "Undeclared"
                    AND b.country_name_short <> "Undeclared"
                    AND a.`{prop_name}` = $prop_value
                    AND b.`{prop_name}` = $prop_value
                RETURN a, cw, b
                """,
                dict(prop_value=prop_value),
            ).get_as_networkx()

            vis.set_labels(compnet_cluster_g, LABEL_PROPS)
            vis.plot(compnet_cluster_g)

        case "map":
            compnet_cluster_df = conn.execute(
                f"""
                MATCH (c:Country)
                WHERE c.country_name_short <> "Undeclared"
                    AND c.`{prop_name}` = $prop_value
                RETURN
                    c.country_iso3_code AS iso3_code,
                    c.`{prop_name}` AS `{prop_name}`
                """,
                dict(prop_value=prop_value),
            ).get_as_df()

            vis.plot_map(compnet_cluster_df, code_col="iso3_code", class_col=prop_name)

In [ ]:
def trade_per_cluster(
    prop_name: str,
    prop_value: int,
    method: Literal["imports", "exports"],
    n: Optional[int] = None,
    debug: bool = False,
) -> pd.DataFrame:
    match method:
        case "exports":
            match_stmt = "MATCH (c:Country)-[ie:Exports]->(p:Product)"
        case "imports":
            match_stmt = "MATCH (c:Country)<-[ie:Imports]-(p:Product)"

    if n is None:
        limit_stmt = ""
        limit_param = dict()
    else:
        limit_stmt = "LIMIT $n"
        limit_param = dict(n=n)

    query = f"""
        {match_stmt}
        WHERE c.country_name_short <> "Undeclared"
            AND c.`{prop_name}` = $prop_value
        RETURN
            p.product_name_short AS product,
            sum(ie.amount_usd) AS total_amount_usd
        ORDER BY total_amount_usd DESC
        {limit_stmt}
    """

    params = dict(prop_value=prop_value) | limit_param

    if debug:
        print_query(query, params)

    products_df = conn.execute(query, params).get_as_df()

    return products_df

Partner clusters are clusters that import what a cluster is exporting. These are likely to match all clusters due to high connectivity in the world economy, but it might not always be the case, depending on the clustering criteria.

In [ ]:
def partner_clusters(
    prop_name: str,
    prop_value: int,
    include_self: bool = True,
    debug: bool = False,
) -> list[int]:
    include_self_stmt = "" if include_self else f"AND c2.`{prop_name}` <> $prop_value"

    query = f"""
        MATCH (c:Country)-[:Exports]-(p:Product)
        MATCH (c2:Country)<-[:Imports]-(p)
        WHERE c.country_name_short <> "Undeclared"
            AND c.`{prop_name}` = $prop_value
            AND c2.`{prop_name}` IS NOT NULL
            {include_self_stmt}
        RETURN DISTINCT c2.`{prop_name}` AS cid
    """

    params = dict(prop_value=prop_value)

    if debug:
        print_query(query, params)

    result = conn.execute(query, params)
    partner_cluster_ids = sorted(c[0] for c in result.get_all())

    return partner_cluster_ids

The following functions will help us compute the intra-cluster and inter-cluster trade alignments, i.e., self-sufficiency and external competitiveness, based on cluster-aggregated market share.

In [ ]:
def trade_alignment_by_cluster(
    prop_name: str,
    prop_value: int,
    method: Literal["intra", "inter"],
) -> pd.DataFrame:
    exports_df = trade_per_cluster(prop_name, prop_value, method="exports")

    match method:
        case "intra":
            imports_df = trade_per_cluster(prop_name, prop_value, method="imports")

        case "inter":
            imports_df = []

            for partner_cid in partner_clusters(prop_name, prop_value):
                partner_imports_df = trade_per_cluster(
                    prop_name,
                    partner_cid,
                    method="imports",
                )
                imports_df.append(partner_imports_df)

            imports_df = pd.concat(imports_df).groupby(["product"]).sum()

        case _:
            raise ValueError(f"method not supported: {method}")

    trade_df = exports_df.merge(
        imports_df,
        on="product",
        how="right" if method == "intra" else "left",
        suffixes=("_exports", "_imports"),
    ).fillna(0)

    trade_df["sdr"] = (
        trade_df.total_amount_usd_exports / trade_df.total_amount_usd_imports
    )

    trade_df = trade_df.sort_values("sdr", ascending=False)

    return trade_df

As a score for measuring either self-sufficiency or external competitiveness, we use weighted average of the Supply-Demand Ration (SDR), where weights are the total export amount (USD) for a given cluster.

In [ ]:
def global_sdr_score(trade_df: pd.DataFrame, eps=1e-9) -> float:
    df = trade_df[~np.isinf(trade_df.sdr)]

    df["log_sdr"] = np.log(np.clip(df.sdr, eps, None))

    weights = df.total_amount_usd_exports
    score = expit((weights * df.log_sdr).sum() / weights.sum())

    return score.item()

### Competing Communities

- Are there any communities representing closely tied competitor clusters?
  - If so, maybe there are specific products per cluster? ☑️
  - If not, we have a global economy that is fairly homogenous and diverse.

For each property computed with the `algo` extension, we'll alter the corresponding node table, recreating the property each time.

In [ ]:
conn.execute(
    """
    ALTER TABLE Country DROP IF EXISTS louvain_id;
    ALTER TABLE Country ADD IF NOT EXISTS louvain_id INT64;

    CALL louvain("compnet")
    WITH node, louvain_id
    SET node.louvain_id = louvain_id;
    """
)

The Louvain method partitions the network by optimizing modularity, which essentially means it will find the best partition of communities within the graph, a community being a dense subgraph, i.e., a subgraph where connections among members are more frequent than to outside nodes.

In [ ]:
compnet_louvain_df = conn.execute(
    """
    MATCH (c:Country)
    WHERE c.country_name_short <> "Undeclared"
    RETURN
        c.node_id AS node_id,
        c.country_name_short AS label,
        c.louvain_id AS louvain_id
    """
).get_as_df()

node_classes = {
    k: g.node_id.to_list() for k, g in compnet_louvain_df.groupby("louvain_id")
}

vis.set_labels(compnet_g, LABEL_PROPS)
vis.plot(compnet_g, node_classes=node_classes, hide_edges=True)

In complex networks, it is not uncommon for a huge community to emerge, along with a low number of moderately large communities, and then a lot of smaller communities. This behavior is not particularly exacerbated here, but it's still visible. Below, we inspect the community size distribution.

In [ ]:
comm_sizes_df = (
    compnet_louvain_df[["louvain_id", "node_id"]]
    .groupby("louvain_id")
    .count()
    .rename(columns=dict(node_id="num_nodes"))
)

comm_sizes_df = comm_sizes_df.reindex(
    comm_sizes_df.num_nodes.sort_values(ascending=False).index
)

comm_sizes_df

In [ ]:
fig, ax = plt.subplots(figsize=(18, 3))
comm_sizes_df.plot.bar(xlabel="Community ID", rot=0, ax=ax)
plt.legend(["No. Nodes"])
plt.show()

Let's also take a look at the members of each community, from largest to smallest.

In [ ]:
for louvain_id in comm_sizes_df.index:
    display(f"LOUVAIN ID: {louvain_id}")
    display(
        compnet_louvain_df[compnet_louvain_df.louvain_id == louvain_id]
        .drop(columns="louvain_id")
        .sort_values("label")
    )

In [ ]:
largest_louvain_id = comm_sizes_df.index[0].item()
largest_louvain_id

In [ ]:
smallest_louvain_id = comm_sizes_df.index[-1].item()
smallest_louvain_id

#### Community Subgraphs

Community subgraphs illustrates clusters where competition is more prevalent among its members than countries outside of the community. For this graph (our Econ CompNet, or `compnet`), they are almost always (if not always) complete subgraphs. We can plot any cluster by its ID.

In [ ]:
plot_cluster("louvain_id", largest_louvain_id)

#### Community Mapping

Network visualization is not always the best approach to understand your data. This is a good example of this. Since we're working with a complete (or nearly complete) subgraph, looking at relationships is less helpful, but looking at a map for a community is a lot more helpful, as we can see below.

In [ ]:
plot_cluster("louvain_id", largest_louvain_id, kind="map")

#### Top Exported Products

- Is there any export overlap between large and small communities?

In [ ]:
largest_comm_top_exported = top_frac(
    trade_per_cluster("louvain_id", largest_louvain_id, method="exports"),
    "total_amount_usd",
)
largest_comm_top_exported

In [ ]:
smallest_comm_top_exported = top_frac(
    trade_per_cluster("louvain_id", smallest_louvain_id, method="exports"),
    "total_amount_usd",
)
smallest_comm_top_exported

In [ ]:
jaccard_sim(largest_comm_top_exported["product"], smallest_comm_top_exported["product"])

#### Top Imported Products

- Is there any import overlap between large and small communities?

In [ ]:
largest_comm_top_imported = top_frac(
    trade_per_cluster("louvain_id", largest_louvain_id, method="imports"),
    "total_amount_usd",
)
largest_comm_top_imported

In [ ]:
smallest_comm_top_imported = top_frac(
    trade_per_cluster("louvain_id", smallest_louvain_id, method="imports"),
    "total_amount_usd",
)
smallest_comm_top_imported

In [ ]:
jaccard_sim(largest_comm_top_imported["product"], smallest_comm_top_imported["product"])

#### Trade Alignment

Trade alignment can be used to determine a cluster's self-sufficiency by looking at internal country-country trade, or it can be used to determine a cluster's external competitiveness by looking at inter-cluster country-country trade. We determine both dimensions of trade alignment (intra and inter cluster) based on the supply/demand ratio, more specifically the weighted average of log-SDR, with weights being total amounts (USD) of exports/imports, globally per cluster.

This score is scaled to a `0..1` range using a sigmoid transformation, so anything above 0.5 should be good. The log-transformation ensures the distribution is not skewed.

##### Self-Sufficiency

Most communities are self-sufficient or nearly self-sufficient, with only community 5 showing a little more vulnerability.

In [ ]:
comm_self_sufficiency_df = pd.DataFrame(
    dict(
        louvain_id=louvain_id,
        score=global_sdr_score(
            trade_alignment_by_cluster(
                "louvain_id",
                louvain_id,
                method="intra",
            )
        ),
    )
    for louvain_id in comm_sizes_df.index
).sort_values("score", ascending=False)

comm_self_sufficiency_df

In [ ]:
colors = comm_self_sufficiency_df.score.apply(
    lambda s: MPL_PALETTE[0] if s >= 0.5 else MPL_PALETTE[1]
)

fig, ax = plt.subplots(figsize=(18, 3))

comm_self_sufficiency_df.plot.bar(
    x="louvain_id",
    y="score",
    xlabel="Community ID",
    color=colors,
    rot=0,
    ax=ax,
)

plt.axhline(y=0.5, color=MPL_PALETTE[1], linestyle="--", linewidth=2)
plt.legend(["Self-Sufficiency Threshold", "Global Log-SDR Score"])
plt.show()

In [ ]:
compnet_louvain_df[compnet_louvain_df.louvain_id == 5]

##### External Competitiveness

Most communities are not particularly competitive externally, but this was to be expected due to the criteria used to cluster—community dense subgraphs also point to higher internal competition.

In [ ]:
comm_external_comp_df = pd.DataFrame(
    dict(
        louvain_id=louvain_id,
        score=global_sdr_score(
            trade_alignment_by_cluster(
                "louvain_id",
                louvain_id,
                method="inter",
            )
        ),
    )
    for louvain_id in comm_sizes_df.index
).sort_values("score", ascending=False)

comm_external_comp_df

In [ ]:
colors = comm_external_comp_df.score.apply(
    lambda s: MPL_PALETTE[0] if s >= 0.5 else MPL_PALETTE[1]
)

fig, ax = plt.subplots(figsize=(18, 3))

comm_external_comp_df.plot.bar(
    x="louvain_id",
    y="score",
    xlabel="Community ID",
    color=colors,
    rot=0,
    ax=ax,
)

plt.axhline(y=0.5, color=MPL_PALETTE[1], linestyle="--", linewidth=2)
plt.legend(["External Competitiveness Threshold", "Global SDR Score"])
plt.show()

In [ ]:
compnet_louvain_df[compnet_louvain_df.louvain_id == 8]

### Weakly Connected Competitors

Strongly connected components in our graph would have capture mutual competition among peers, cyclical or balanced rivalries, or equivalent strategic positions. However, once we removed the "Undeclared" pseudo-country, we weren't able to find any strongly connected components that were not singletons.

As such, we compute the weakly connected components, instead capturing the individual or isolated components of countries competing among themselves, regardless of export amount (which establishes direction, in our graph).

In [ ]:
conn.execute(
    """
    ALTER TABLE Country DROP IF EXISTS wcc_id;
    ALTER TABLE Country ADD IF NOT EXISTS wcc_id INT64;

    CALL weakly_connected_components("compnet")
    WITH node, group_id
    SET node.wcc_id = group_id;
    """
)

In [ ]:
compnet_wcc_df = conn.execute(
    """
    MATCH (c:Country)
    WHERE c.country_name_short <> "Undeclared"
    RETURN c.node_id AS node_id, c.country_name_short AS label, c.wcc_id AS wcc_id
    """
).get_as_df()

node_classes = {k: g.node_id.to_list() for k, g in compnet_wcc_df.groupby("wcc_id")}

vis.set_labels(compnet_g, LABEL_PROPS)
vis.plot(compnet_g, node_classes=node_classes, hide_edges=True)

As we can see, there a multiple weakly connected competitors, but most of them are single nodes in their own SCC. Other than that, there is a large component of 64 countries, and then two other smaller components with over 20 nodes each, that we'll inspect below.

In [ ]:
wcc_sizes_df = (
    compnet_wcc_df[["wcc_id", "node_id"]]
    .groupby("wcc_id")
    .count()
    .rename(columns=dict(node_id="num_nodes"))
)

wcc_sizes_df = wcc_sizes_df.reindex(
    wcc_sizes_df.num_nodes.sort_values(ascending=False).index
)

wcc_sizes_df

In [ ]:
wcc_sizes_ord_df = wcc_sizes_df.reset_index(drop=True)

wcc_singleton_threshold = (
    wcc_sizes_ord_df[wcc_sizes_ord_df.num_nodes <= 1].index[0].item()
)

fig, ax = plt.subplots(figsize=(30, 5))

wcc_sizes_df.plot.bar(rot=0, ax=ax)

plt.axvline(
    x=wcc_singleton_threshold,
    color=MPL_PALETTE[1],
    linestyle="--",
    linewidth=2,
)

plt.legend(["Singleton Threshold", "No. Nodes"])

plt.show()

Let's take a look at the members of each weak component, from largest to smallest.

In [ ]:
for wcc_id in wcc_sizes_df[wcc_sizes_df.num_nodes > 1].index:
    display(f"WCC ID: {wcc_id}")
    display(compnet_wcc_df[compnet_wcc_df.wcc_id == wcc_id].drop(columns="wcc_id"))

In [ ]:
largest_wcc_id = wcc_sizes_df.index[0].item()
largest_wcc_id

In [ ]:
smallest_wcc_id = wcc_sizes_df.index[-1].item()
smallest_wcc_id

#### Component Subgraphs

In [ ]:
plot_cluster("wcc_id", largest_wcc_id)

#### Component Mapping

In [ ]:
plot_cluster("wcc_id", largest_wcc_id, kind="map")

#### Top Exported Products

- Is there any export overlap between large and small components?

In [ ]:
largest_wcc_top_exported = top_frac(
    trade_per_cluster("wcc_id", largest_wcc_id, "exports"),
    "total_amount_usd",
)
largest_wcc_top_exported

In [ ]:
smallest_wcc_top_exported = top_frac(
    trade_per_cluster("wcc_id", smallest_wcc_id, "exports"),
    "total_amount_usd",
)
smallest_wcc_top_exported

In [ ]:
jaccard_sim(largest_wcc_top_exported["product"], smallest_wcc_top_exported["product"])

#### Top Imported Products

- Is there any import overlap between large and small components?

In [ ]:
largest_wcc_top_imported = top_frac(
    trade_per_cluster("wcc_id", largest_wcc_id, "imports"),
    "total_amount_usd",
)
largest_wcc_top_imported

In [ ]:
smallest_wcc_top_imported = top_frac(
    trade_per_cluster("wcc_id", smallest_wcc_id, "imports"),
    "total_amount_usd",
)
smallest_wcc_top_imported

In [ ]:
jaccard_sim(largest_comm_top_imported["product"], smallest_wcc_top_imported["product"])

#### Trade Alignment

Again, trade alignment can be used to determine a cluster's self-sufficiency by looking at internal country-country trade, or it can be used to determine a cluster's external competitiveness by looking at inter-cluster country-country trade. We determine both dimensions of trade alignment (intra and inter cluster) based on the supply/demand ratio, more specifically the weighted average of log-SDR, with weights being total amounts (USD) of exports/imports, globally per cluster.

This score is scaled to a `0..1` range using a sigmoid transformation, so anything above 0.5 should be good. The log-transformation ensures the distribution is not skewed.

##### Self-Sufficiency

Most components are self-sufficient or nearly self-sufficient, with only three of them, components 209, 22 and 196, showing a little more vulnerability.

In [ ]:
wcc_self_sufficiency_df = pd.DataFrame(
    dict(
        wcc_id=wcc_id,
        score=global_sdr_score(
            trade_alignment_by_cluster(
                "wcc_id",
                wcc_id,
                method="intra",
            )
        ),
    )
    for wcc_id in wcc_sizes_df.index
).sort_values("score", ascending=False)

wcc_self_sufficiency_df

In [ ]:
colors = wcc_self_sufficiency_df.score.apply(
    lambda s: MPL_PALETTE[0] if s >= 0.5 else MPL_PALETTE[1]
)

fig, ax = plt.subplots(figsize=(30, 5))

wcc_self_sufficiency_df.plot.bar(
    x="wcc_id",
    y="score",
    xlabel="Weak Component ID",
    color=colors,
    rot=0,
    ax=ax,
)

plt.axhline(y=0.5, color=MPL_PALETTE[1], linestyle="--", linewidth=2)
plt.legend(["Self-Sufficiency Threshold", "Global SDR Score"])
plt.show()

In [ ]:
compnet_wcc_df[compnet_wcc_df.wcc_id == 0]

##### External Competitiveness

Most components are not particularly competitive externally, even less so than communities, with the large majority having a SDR-based score lower than 0.1.

In [ ]:
wcc_external_comp_df = pd.DataFrame(
    dict(
        wcc_id=wcc_id,
        score=global_sdr_score(
            trade_alignment_by_cluster(
                "wcc_id",
                wcc_id,
                method="inter",
            )
        ),
    )
    for wcc_id in wcc_sizes_df.index
).sort_values("score", ascending=False)

wcc_external_comp_df

In [ ]:
colors = wcc_external_comp_df.score.apply(
    lambda s: MPL_PALETTE[0] if s >= 0.5 else MPL_PALETTE[1]
)

fig, ax = plt.subplots(figsize=(30, 5))

wcc_external_comp_df.plot.bar(
    x="wcc_id",
    y="score",
    xlabel="Weak Component ID",
    color=colors,
    rot=0,
    ax=ax,
)

plt.axhline(y=0.5, color=MPL_PALETTE[1], linestyle="--", linewidth=2)
plt.legend(["External Competitiveness Threshold", "Global SDR Score"])
plt.show()

In [ ]:
compnet_louvain_df[compnet_louvain_df.louvain_id == 0]

### Comparing Communities and Components

By matching the clustering (communities and weak components) with the highest number of clusters, and therefore smaller clusters, to the clustering with the lowest number of clusters, we can run a pairwise cluster comparison:

- Which countries belong to a community, but not the weak component?
- Which countries belong to a weak component, but not the community?
- Which countries belong to both?
- Is there a particular semantic to these countries?

In [ ]:
len(wcc_sizes_df), len(comm_sizes_df)

#### NN-Clusters

We compute community to weak component similarities, selecting the nearest-neighbor community for each component. Given the higher number of components when compared to communities, we'll necessarily have repeated nearest-neighbor communities.

In [ ]:
cluster_sim_df = []

for wcc_id, wcc in compnet_wcc_df.groupby("wcc_id"):
    for louvain_id, comm in compnet_louvain_df.groupby("louvain_id"):
        cluster_sim_df.append(
            dict(
                wcc_id=wcc_id,
                louvain_id=louvain_id,
                sim=jaccard_sim(wcc.label, comm.label),
            )
        )

cluster_sim_df = pd.DataFrame(cluster_sim_df)
cluster_sim_df = cluster_sim_df.loc[cluster_sim_df.groupby(["wcc_id"]).idxmax().sim]
cluster_sim_df

For example, community 5 matches with 20 different weak components.

In [ ]:
cluster_sim_df.louvain_id.value_counts()

In [ ]:
cluster_sim_df[cluster_sim_df.louvain_id == 5]

#### Set Comparison

Let's select a weakest component and retrieve its NN community to compare.

In [ ]:
# comp_wcc_id = largest_wcc_id
comp_wcc_id = compnet_wcc_df.loc[compnet_wcc_df.label == "Australia", "wcc_id"].item()

comp_comm_id = cluster_sim_df.loc[
    cluster_sim_df.wcc_id == comp_wcc_id,
    "louvain_id",
].item()

comp_wcc_id, comp_comm_id

In [ ]:
comp_wcc_countries = set(
    compnet_wcc_df.loc[compnet_wcc_df.wcc_id == comp_wcc_id, "label"]
)

comp_louvain_countries = set(
    compnet_louvain_df.loc[compnet_louvain_df.louvain_id == comp_comm_id, "label"]
)

##### WCC Exclusive

In [ ]:
pd.Series(
    list(comp_wcc_countries - comp_louvain_countries),
    name="country",
).sort_values().to_frame()

##### Community Exclusive

In [ ]:
pd.Series(
    list(comp_louvain_countries - comp_wcc_countries),
    name="country",
).sort_values().to_frame()

##### WCC and Community Overlap

In [ ]:
pd.Series(
    list(comp_wcc_countries | comp_louvain_countries),
    name="country",
).sort_values().to_frame()

### Economical Pressure (PageRank)

Economical pressure can easily be measured using PageRank, as it is a converging metric that aggregates the overall incoming competition strength, increasing its value as the contributing competing countries are themselves under economical pressure.

In [ ]:
conn.execute(
    """
    ALTER TABLE Country DROP IF EXISTS pagerank;
    ALTER TABLE Country ADD IF NOT EXISTS pagerank DOUBLE;

    CALL page_rank("compnet", maxIterations := 100)
    WITH node, rank
    SET node.pagerank = rank
    """
)

#### Most Pressured Countries

In [ ]:
most_pressured_df = conn.execute(
    """
    MATCH (c:Country)
    WHERE c.country_name_short <> "Undeclared"
    RETURN
        c.node_id AS node_id,
        c.country_name_short AS label,
        c.pagerank AS pagerank
    ORDER BY c.pagerank DESC
    LIMIT 25
    """
).get_as_df()

fig, ax = plt.subplots(figsize=(5, 8))
most_pressured_df.iloc[::-1].plot.barh(x="label", y="pagerank", ax=ax)
plt.ylabel(None)
plt.legend([])
plt.show()

#### Least Pressured Countries

In [ ]:
least_pressured_df = conn.execute(
    """
    MATCH (c:Country)
    WHERE c.country_name_short <> "Undeclared"
    RETURN
        c.node_id AS node_id,
        c.country_name_short AS label,
        c.pagerank AS pagerank
    ORDER BY c.pagerank ASC
    LIMIT 25
    """
).get_as_df()

fig, ax = plt.subplots(figsize=(5, 8))
least_pressured_df.iloc[::-1].plot.barh(x="label", y="pagerank", ax=ax)
plt.ylabel(None)
plt.title("Least Economically Pressured Countries (PageRank)", loc="right")
plt.legend([])
plt.show()

# Closing Remarks

Economies are complex systems, and the complex relations between markets can be captured using a graph. Determining which nodes and relationships to model is crucial to interpretation—our graph focused on competition relationships, and so our metrics and partition approaches illustrated this.

Network analysis tools are usually not as exotic as they want to make us believe. Useful graph data science is usually not that complex, particularly now that tooling is widely available, but it can certainly be extremely insightful, specially when the graph is correctly modeled.

This is only a small introduction to this topic, using world economy and trade as an example topic, which I have been particularly interested in.

The economy and the world overall is suffering. Graphs will help us find solution to complex problems, but it requires the commitment to always ask yourself: could I do this without a graph? When the answer is yes, then you should rethink your approach. If you're not looking at complex relations, you're just doing more of the same.

Bottom line, use graphs and use them correctly.